In [ ]:
!pip install pytesseract

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
ls /content/drive

MyDrive/


In [ ]:
import cv2
import numpy as np
import pytesseract


# ---------------------------
# 1) 4점 정렬
# ---------------------------
def order_points(pts):
    rect = np.zeros((4, 2), dtype="float32")
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]      # TL
    rect[2] = pts[np.argmax(s)]      # BR

    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]   # TR
    rect[3] = pts[np.argmax(diff)]   # BL

    return rect


# ---------------------------
# 2) 문서 윤곽(사각형) 찾기
# ---------------------------
def find_document_contour(image, debug=False):
    orig = image.copy()
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    gray = cv2.GaussianBlur(gray, (7, 7), 0)
    edges = cv2.Canny(gray, 50, 150)

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    edges = cv2.dilate(edges, kernel, iterations=2)

    cnts, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL,
                               cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        raise Exception("윤곽을 찾지 못했습니다.")

    cnts = sorted(cnts, key=cv2.contourArea, reverse=True)
    doc_cnt = None

    for c in cnts:
        peri = cv2.arcLength(c, True)
        approx = cv2.approxPolyDP(c, 0.015 * peri, True)
        if len(approx) == 4:
            doc_cnt = approx.reshape(4, 2)
            break

    # fallback
    if doc_cnt is None:
        c = cnts[0]
        rect = cv2.minAreaRect(c)
        box = cv2.boxPoints(rect)
        doc_cnt = np.int32(box)

    # 윤곽 정렬(가로/세로 뒤틀림 방지)
    doc_cnt = cv2.convexHull(doc_cnt).reshape(-1, 2)

    if debug:
        dbg = orig.copy()
        cv2.drawContours(dbg, [doc_cnt.astype(int)], -1, (0, 0, 255), 3)
        cv2.imwrite("debug_doc_contour.png", dbg)

    return doc_cnt


# ---------------------------
# 3) 원근 보정
# ---------------------------
def four_point_transform(image, pts):
    rect = order_points(pts.astype("float32"))
    (tl, tr, br, bl) = rect

    widthA = np.linalg.norm(br - bl)
    widthB = np.linalg.norm(tr - tl)
    maxWidth = int(max(widthA, widthB))

    heightA = np.linalg.norm(tr - br)
    heightB = np.linalg.norm(tl - bl)
    maxHeight = int(max(heightA, heightB))

    dst = np.array([
        [0, 0],
        [maxWidth - 1, 0],
        [maxWidth - 1, maxHeight - 1],
        [0, maxHeight - 1]], dtype="float32")

    M = cv2.getPerspectiveTransform(rect, dst)
    warped = cv2.warpPerspective(image, M, (maxWidth, maxHeight))
    return warped


# ---------------------------
# 세로 방향 보정 (가로 → 세로)
# ---------------------------
def fix_vertical(img):
    h, w = img.shape[:2]
    if w > h:
        img = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
    return img


# ---------------------------
# **뒤집힘(180도) 감지 — 텍스트 밀도 기반**
# ---------------------------
def is_upside_down_ocr(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    data = pytesseract.image_to_data(gray, output_type=pytesseract.Output.DICT)

    ys = []

    for i in range(len(data["text"])):
        if data["text"][i].strip() != "":
            ys.append(data["top"][i] + data["height"][i] / 2)

    if len(ys) == 0:
        return False

    avg_y = np.mean(ys)
    h = img.shape[0]

    # 평균 y가 위쪽에 가까우면 뒤집힌 것
    return avg_y < h * 0.45


def fix_upside_down(img):
    if is_upside_down_ocr(img):
        print("뒤집힘 감지 → 180도 회전")
        return cv2.rotate(img, cv2.ROTATE_180)
    else:
        print("정상 방향 유지")
        return img



# ---------------------------
# 4) 전체 파이프라인
# ---------------------------
def process_receipt(path, debug=False):
    image = cv2.imread(path)
    if image is None:
        raise FileNotFoundError(path)

    # 1) 문서 윤곽 찾기
    doc_cnt = find_document_contour(image, debug=debug)

    # 2) 원근 보정
    warped = four_point_transform(image, doc_cnt)

    # 3) 세로 정렬
    warped = fix_vertical(warped)

    # 4) 뒤집힘 보정
    warped = fix_upside_down(warped)

    # 최종 출력 이미지
    return warped


# ---------------------------
# 사용 예시
out = process_receipt("/content/drive/MyDrive/receipt3_rot.jpg", debug=True)
cv2.imwrite("fixed_receipt.png", out)
print("저장 완료: fixed_receipt.png")


정상 방향 유지
저장 완료: fixed_receipt.png
